In [3]:
import os
import json
import torch
from PIL import Image
from peft import PeftModel
from llava.model.builder import load_pretrained_model
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX

# ==================== Configuration ====================
SFT_PATH = "../../LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/sft_model"
LORA_PATH = "../../LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/rlhf_lora_adapter_model"
IMAGE_DIR = "../test_images"
OUTPUT_JSON = "cross_image_probe_results.json"

In [4]:
# 16 cross-image probes (schema v1.0) + shared benchmark helpers
PROBES = [
    {
        "image": "test.jpg",
        "id": "T1",
        "type": "existence",
        "question": "Is there a keyboard in this image? Answer yes or no.",
        "gt": "no",
        "note": "Keyboard is plausible in an office scene but is not present"
    },
    {
        "image": "test.jpg",
        "id": "T2",
        "type": "count",
        "question": "How many tissue boxes are in this image? Answer with a number.",
        "gt": "1",
        "note": "Exactly one tissue box; SFT+ answered correctly, RLHF once regressed to 2"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C1",
        "type": "existence",
        "question": "Is there a coffee machine in this image? Answer yes or no.",
        "gt": "no",
        "note": "Coffee machine is common in kitchens but is not in the image"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C2",
        "type": "existence",
        "question": "Is there a banana in the bowl? Answer yes or no.",
        "gt": "no",
        "note": "Fruit bowl contains only apples and pears"
    },
    {
        "image": "clean_counter.jpg",
        "id": "C3",
        "type": "relation",
        "question": "Is the bowl of fruit to the left or right of the sink? Answer left or right.",
        "gt": "left",
        "note": "Fruit bowl is to the left of the sink"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E1",
        "type": "existence",
        "question": "Is there a keyboard on the desk? Answer yes or no.",
        "gt": "no",
        "note": "Keyboard is typical desk clutter but is not present"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E2",
        "type": "existence",
        "question": "Is there a computer monitor in this image? Answer yes or no.",
        "gt": "no",
        "note": "Monitor is common on desks but is not in the image"
    },
    {
        "image": "empty_desk.jpg",
        "id": "E3",
        "type": "count",
        "question": "How many cups are in this image? Answer with a number.",
        "gt": "1",
        "note": "Only one gray mug on the desk"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K1",
        "type": "existence",
        "question": "Is there a cat in this image? Answer yes or no.",
        "gt": "no",
        "note": "Drawing shows a dog, not a cat"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K2",
        "type": "count",
        "question": "How many people are in this image? Answer with a number.",
        "gt": "5",
        "note": "Five stick-figure people"
    },
    {
        "image": "kids_drawing.jpg",
        "id": "K3",
        "type": "attribute",
        "question": "What color is the dress of the person on the far left? Answer with one color.",
        "gt": "yellow",
        "note": "Leftmost figure wears a yellow dress"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O1",
        "type": "relation",
        "question": "Is the paper boat in front of or behind the pen holder? Answer front or behind.",
        "gt": "front",
        "note": "Paper boat is in front of the pen holder"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O2",
        "type": "relation",
        "question": "Is the pink pen in front of or behind the paper boat? Answer front or behind.",
        "gt": "front",
        "note": "Pink pen is in front of the paper boat"
    },
    {
        "image": "objects_overlapping.jpg",
        "id": "O3",
        "type": "existence",
        "question": "Is there a yellow sticky note on the wall? Answer yes or no.",
        "gt": "yes",
        "note": "Yellow sticky note is visible on the background wall"
    },
    {
        "image": "abstract_painting.jpg",
        "id": "A1",
        "type": "existence",
        "question": "Is there a tree in this image? Answer yes or no.",
        "gt": "no",
        "note": "Pure abstract painting with no tree"
    },
    {
        "image": "abstract_painting.jpg",
        "id": "A2",
        "type": "open",
        "question": "Describe this image in detail.",
        "gt": "abstract",
        "note": "Open-ended: check whether the model invents natural-scene objects"
    }
]

import json

SCHEMA_VERSION = "1.0"
BENCHMARK_NAME = "cross_image_hallucination_probe"

NATURAL_OBJECTS = [
    "mountain", "lake", "tree", "forest", "person", "people",
    "man", "woman", "dog", "cat", "house", "building", "car", "sky", "river",
]


def judge_correctness(pred, gt, qtype):
    """Score a model answer against ground truth using loose heuristics.

    Args:
        pred: Raw model response text.
        gt: Ground-truth answer string from the probe definition.
        qtype: Probe type — existence, count, relation, attribute, or open.

    Returns:
        True if the answer passes the probe's correctness check.
    """
    pred_lower = pred.lower().strip(".")
    gt_lower = gt.lower().strip()
    if qtype == "open":
        return not any(obj in pred_lower for obj in NATURAL_OBJECTS)
    if gt_lower in ["yes", "no"]:
        if gt_lower == "yes":
            return "yes" in pred_lower and "no" not in pred_lower
        return "no" in pred_lower and "yes" not in pred_lower
    return gt_lower in pred_lower


def _verdict_label(correct, qtype):
    """Map a boolean correctness flag to a JSON verdict string.

    Args:
        correct: Whether judge_correctness returned True.
        qtype: Probe type; open probes use hallucination-specific labels.

    Returns:
        Verdict label, e.g. "correct", "wrong", or "natural_object_hallucination".
    """
    if qtype == "open":
        return "no_natural_object_hallucination" if correct else "natural_object_hallucination"
    return "correct" if correct else "wrong"


def build_benchmark_output(run_id, models, probes, answers_by_model, pairwise=None):
    """Build the standardized schema v1.0 benchmark JSON payload.

    Args:
        run_id: Identifier for this benchmark run (e.g. "llava_onevision").
        models: List of dicts with key, name, and path for each evaluated model.
        probes: Probe definitions (the PROBES list).
        answers_by_model: Dict mapping model key -> {probe_id: answer_text}.
        pairwise: Optional two model keys for head-to-head summary stats.

    Returns:
        Dict ready to serialize as cross-image probe results JSON.
    """
    model_keys = [m["key"] for m in models]
    per_probe = []
    for probe in probes:
        answers, correct, verdict = {}, {}, {}
        for key in model_keys:
            ans = answers_by_model[key].get(probe["id"], "")
            ok = judge_correctness(ans, probe["gt"], probe["type"])
            answers[key] = ans
            correct[key] = ok
            verdict[key] = _verdict_label(ok, probe["type"])
        per_probe.append({**probe, "answers": answers, "correct": correct, "verdict": verdict})

    by_model = {}
    for key in model_keys:
        c = sum(1 for r in per_probe if r["correct"][key])
        t = len(per_probe)
        by_type = {}
        open_halluc = open_ok = 0
        for r in per_probe:
            q = r["type"]
            by_type.setdefault(q, {"correct": 0, "wrong": 0})
            if r["correct"][key]:
                by_type[q]["correct"] += 1
            else:
                by_type[q]["wrong"] += 1
            if q == "open":
                if r["correct"][key]:
                    open_ok += 1
                else:
                    open_halluc += 1
        by_model[key] = {
            "correct": c, "wrong": t - c, "accuracy": round(c / max(t, 1), 4),
            "open_halluc": open_halluc, "open_no_halluc": open_ok, "by_type": by_type,
        }

    summary = {"total": len(per_probe), "by_model": by_model, "pairwise": None}
    if pairwise and len(pairwise) == 2:
        left, right = pairwise
        pw = {"both_correct": 0, "both_wrong": 0, f"{left}_better": 0, f"{right}_better": 0}
        for r in per_probe:
            if r["type"] == "open":
                continue
            l_ok, r_ok = r["correct"][left], r["correct"][right]
            if l_ok and r_ok:
                pw["both_correct"] += 1
            elif not l_ok and not r_ok:
                pw["both_wrong"] += 1
            elif l_ok:
                pw[f"{left}_better"] += 1
            else:
                pw[f"{right}_better"] += 1
        summary["pairwise"] = pw

    return {
        "schema_version": SCHEMA_VERSION,
        "benchmark": BENCHMARK_NAME,
        "run_id": run_id,
        "models": models,
        "probes": probes,
        "per_probe": per_probe,
        "summary": summary,
    }


def save_benchmark_output(output, path):
    """Write benchmark output dict to a JSON file with UTF-8 encoding.

    Args:
        output: Schema v1.0 payload from build_benchmark_output.
        path: Destination file path.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)


In [5]:
def generate_answer(model, tokenizer, image_processor, image_path, question):
    """Run greedy LLaVA generation for one (image, question) pair.

    Args:
        model: Loaded LLaVA vision-language model.
        tokenizer: Model tokenizer with image-token support.
        image_processor: LLaVA image preprocessor.
        image_path: Path to the probe image file.
        question: Probe question text.

    Returns:
        Decoded assistant response (special tokens stripped).
    """
    image = Image.open(image_path).convert("RGB")
    images_tensor = process_images([image], image_processor, model.config)
    images_tensor = images_tensor.to(model.device, dtype=torch.float16)

    prompt = f"USER: {DEFAULT_IMAGE_TOKEN}\n{question}\nASSISTANT:"
    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0)
    input_ids = input_ids.to(model.device)

    # Use a longer budget for open-ended description probes.
    max_new_tokens = 128 if question.startswith("Describe") else 32

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            images=images_tensor,
            do_sample=False,
            temperature=0.2,
            max_new_tokens=max_new_tokens,
            use_cache=False,
        )

    return tokenizer.decode(output_ids[0, input_ids.shape[1]:], skip_special_tokens=True).strip()



In [6]:
# ==================== Phase 1: SFT+ base model ====================
# Evaluate the supervised fine-tuned checkpoint before loading any RLHF adapter.
print("=" * 80)
print("PHASE 1: Loading SFT+ base model...")
print("=" * 80)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=SFT_PATH, model_base=None, model_name="llava-v1.5-7b",
    load_8bit=True, device_map="auto",
)
model = model.eval()

print("\nRunning SFT+ on all probes...")
results_sft = {}
for p in PROBES:
    img_path = os.path.join(IMAGE_DIR, p["image"])
    if not os.path.exists(img_path):
        print(f"[SKIP] {img_path} not found")
        continue

    ans = generate_answer(model, tokenizer, image_processor, img_path, p["question"])
    results_sft[p["id"]] = ans
    print(f"[SFT+][{p['id']}] {p['image'][:15]:<<15} | {ans[:50]}")

PHASE 1: Loading SFT+ base model...


/data/workspaces/zwang829/conda_envs/llava_rlhf/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.12s/it]
Some weights of the model checkpoint at ../../LLaVA-RLHF/models/LLaVA-RLHF-7B-v1.5-224/sft_model were not used when initializing LlavaLlamaForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.22.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.7.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.1.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.21.mlp.fc2.bias', 'mo


Running SFT+ on all probes...
[SFT+][T1] test.jpg<<<<<<< | Yes
[SFT+][T2] test.jpg<<<<<<< | 1
[SFT+][C1] clean_counter.j | No
[SFT+][C2] clean_counter.j | Yes
[SFT+][C3] clean_counter.j | Left
[SFT+][E1] empty_desk.jpg< | No
[SFT+][E2] empty_desk.jpg< | No
[SFT+][E3] empty_desk.jpg< | 1
[SFT+][K1] kids_drawing.jp | No
[SFT+][K2] kids_drawing.jp | 6
[SFT+][K3] kids_drawing.jp | Pink
[SFT+][O1] objects_overlap | Behind
[SFT+][O2] objects_overlap | Behind
[SFT+][O3] objects_overlap | Yes
[SFT+][A1] abstract_painti | No
[SFT+][A2] abstract_painti | The image is a colorful and abstract painting, rem


In [7]:
# ==================== Phase 2: RLHF (LoRA adapter) ====================
# Reuse the same SFT backbone and attach the RLHF LoRA weights on top.
print("\n" + "=" * 80)
print("PHASE 2: Loading LoRA adapter (RLHF)...")
print("=" * 80)

model = PeftModel.from_pretrained(model, LORA_PATH, device_map="auto")
model = model.eval()

print("\nRunning RLHF on all probes...")
results_rlhf = {}
for p in PROBES:
    img_path = os.path.join(IMAGE_DIR, p["image"])
    if not os.path.exists(img_path):
        continue

    ans = generate_answer(model, tokenizer, image_processor, img_path, p["question"])
    results_rlhf[p["id"]] = ans
    print(f"[RLHF][{p['id']}] {p['image'][:15]:<<15} | {ans[:50]}")


PHASE 2: Loading LoRA adapter (RLHF)...

Running RLHF on all probes...
[RLHF][T1] test.jpg<<<<<<< | Yes, there is a keyboard in this image. The keyboa
[RLHF][T2] test.jpg<<<<<<< | There are two tissue boxes in this image.
[RLHF][C1] clean_counter.j | No, there is no coffee machine in this image. The 
[RLHF][C2] clean_counter.j | Yes
[RLHF][C3] clean_counter.j | The bowl of fruit is to the left of the sink.
[RLHF][E1] empty_desk.jpg< | No
[RLHF][E2] empty_desk.jpg< | No, there is no computer monitor in this image. Th
[RLHF][E3] empty_desk.jpg< | There are two cups in this image.
[RLHF][K1] kids_drawing.jp | No, there is no cat in this image. The image featu
[RLHF][K2] kids_drawing.jp | There are six people in this image.
[RLHF][K3] kids_drawing.jp | Yellow
[RLHF][O1] objects_overlap | The paper boat is in front of the pen holder.
[RLHF][O2] objects_overlap | The pink pen is in front of the paper boat.
[RLHF][O3] objects_overlap | Yes, there is a yellow sticky note on the wall.
[RLHF][A

In [8]:
benchmark_output = build_benchmark_output(
    run_id="llava_rlhf_sft_rlhf",
    models=[
        {"key": "sft", "name": "LLaVA-RLHF SFT+", "path": SFT_PATH},
        {"key": "rlhf", "name": "LLaVA-RLHF RLHF", "path": LORA_PATH},
    ],
    probes=PROBES,
    answers_by_model={"sft": results_sft, "rlhf": results_rlhf},
    pairwise=["sft", "rlhf"],
)
per_probe = benchmark_output["per_probe"]
summary = benchmark_output["summary"]

# ==================== Results table ====================
print("\n" + "=" * 100)
print("CROSS-IMAGE HALLUCINATION PROBE RESULTS")
print("=" * 100)
print(f"{'ID':<4} {'Image':<15} {'Type':<12} {'Ground Truth':<10} {'SFT+':<25} {'RLHF':<25} {'Verdict'}")
print("-" * 100)

for row in per_probe:
    sid = row["id"]
    sft_ans = row["answers"]["sft"]
    rlhf_ans = row["answers"]["rlhf"]
    gt = row["gt"]
    sft_correct = row["correct"]["sft"]
    rlhf_correct = row["correct"]["rlhf"]

    if row["type"] == "open":
        verdict = f"SFT+ halluc={not sft_correct}, RLHF halluc={not rlhf_correct}"
    elif sft_correct and rlhf_correct:
        verdict = "Both correct"
    elif not sft_correct and not rlhf_correct:
        verdict = "Both wrong"
    elif sft_correct and not rlhf_correct:
        verdict = "RLHF worse"
    else:
        verdict = "RLHF better"

    print(f"{sid:<4} {row['image'][:14]:<15} {row['type']:<12} {gt:<10} {sft_ans[:24]:<25} {rlhf_ans[:24]:<25} {verdict}")

print("-" * 100)


CROSS-IMAGE HALLUCINATION PROBE RESULTS
ID   Image           Type         Ground Truth SFT+                      RLHF                      Verdict
----------------------------------------------------------------------------------------------------
T1   test.jpg        existence    no         Yes                       Yes, there is a keyboard  Both wrong
T2   test.jpg        count        1          1                         There are two tissue box  RLHF worse
C1   clean_counter.  existence    no         No                        No, there is no coffee m  Both correct
C2   clean_counter.  existence    no         Yes                       Yes                       Both wrong
C3   clean_counter.  relation     left       Left                      The bowl of fruit is to   Both correct
E1   empty_desk.jpg  existence    no         No                        No                        Both correct
E2   empty_desk.jpg  existence    no         No                        No, there is no computer  

In [9]:
sft_stats = summary["by_model"]["sft"]
rlhf_stats = summary["by_model"]["rlhf"]
pairwise = summary["pairwise"]

print(
    f"\n[Open-ended Abstract] SFT+ hallucination count: {sft_stats['open_halluc']}, "
    f"RLHF: {rlhf_stats['open_halluc']}"
)
print(
    f"\nSummary: Both correct={pairwise['both_correct']}, Both wrong={pairwise['both_wrong']}, "
    f"SFT+ better={pairwise['sft_better']}, RLHF better={pairwise['rlhf_better']}"
)
print(
    f"Accuracy: SFT+={sft_stats['accuracy']:.1%}, RLHF={rlhf_stats['accuracy']:.1%}"
)
print("=" * 100)

save_benchmark_output(benchmark_output, OUTPUT_JSON)
print(f"\n[OK] Standardized results saved to {OUTPUT_JSON}")


[Open-ended Abstract] SFT+ hallucination count: 1, RLHF: 1

Summary: Both correct=6, Both wrong=3, SFT+ better=3, RLHF better=3
Accuracy: SFT+=56.2%, RLHF=56.2%

[OK] Standardized results saved to cross_image_probe_results.json
